In [6]:
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings('ignore')

CLEAN_DIR = '../data/clean'
DASH_DIR  = '../data/dashboard'
os.makedirs(DASH_DIR, exist_ok=True)

# ── Safe loader ──────────────────────────────────────────────
# Reads CSV, ensures 'year' column always exists.
# Datasets with a 'date' column (cpi_headline, cpi_inflation,
# cpi_state, cpi_lowincome) get 'year' extracted automatically.
def load_clean(name):
    path = f'{CLEAN_DIR}/{name}.csv'
    if not os.path.exists(path):
        print(f'  ❌  MISSING: {path}')
        print(f'      → Run 00_downloadDataFinal.ipynb then 01_EDA___Data_Cleaning.ipynb first!')
        return None
    df = pd.read_csv(path)
    # Ensure 'year' column exists (monthly CPI files use 'date')
    if 'year' not in df.columns and 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df['year'] = df['date'].dt.year
    return df

print('Loading 21 clean datasets...')
print('=' * 55)

# ── TEMA 1 — Demografi ────────────────────────────────────────
print('\n  📦  TEMA 1 — Demografi')
pop_malaysia    = load_clean('population_malaysia')    # 1970–2024 | age × sex × ethnicity
pop_state       = load_clean('population_state')       # 1970–2024 | by state
pop_district    = load_clean('population_district')    # 2000–2024 | by district
fertility       = load_clean('fertility')              # 1958–2023 | national TFR
fertility_state = load_clean('fertility_state')        # 1980–2023 | TFR by state
births          = load_clean('births_annual')          # 2000–2023 | live births by sex
deaths          = load_clean('deaths')                 # 2000–2023 | deaths by sex
marriages       = load_clean('marriages')              # 2017–2022 | marriages
hh_profile      = load_clean('hh_profile')            # HIES cycles | HH size/composition

# ── TEMA 2 — Kos Sara Hidup ───────────────────────────────────
print('\n  📦  TEMA 2 — Kos Sara Hidup')
hh_income           = load_clean('hh_income')              # 1970–2022 | national median/mean income
hh_income_state     = load_clean('hh_income_state')        # 1970–2022 | income by state
hh_inequality       = load_clean('hh_inequality')          # 1970–2022 | national Gini coefficient
hh_inequality_state = load_clean('hh_inequality_state')    # 1970–2022 | Gini by state
hh_poverty          = load_clean('hh_poverty')             # 1970–2022 | national poverty rates
hh_poverty_state    = load_clean('hh_poverty_state')       # 1970–2022 | poverty by state
hies_state          = load_clean('hies_state')             # HIES cycles | expenditure by state
cpi_annual          = load_clean('cpi_annual')             # 1970–2024 | annual CPI index
cpi_headline        = load_clean('cpi_headline')           # 2010–2025 | monthly CPI index
cpi_inflation       = load_clean('cpi_headline_inflation') # 2010–2025 | monthly YoY inflation
cpi_state           = load_clean('cpi_state')              # 2010–2025 | CPI by state
cpi_lowincome       = load_clean('cpi_lowincome')          # 2010–2025 | CPI for low-income HH

# ── Summary ───────────────────────────────────────────────────
all_vars = [
    pop_malaysia, pop_state, pop_district, fertility, fertility_state,
    births, deaths, marriages, hh_profile,
    hh_income, hh_income_state, hh_inequality, hh_inequality_state,
    hh_poverty, hh_poverty_state, hies_state, cpi_annual,
    cpi_headline, cpi_inflation, cpi_state, cpi_lowincome
]

all_names = [
    'population_malaysia', 'population_state', 'population_district',
    'fertility', 'fertility_state', 'births_annual', 'deaths', 'marriages', 'hh_profile',
    'hh_income', 'hh_income_state', 'hh_inequality', 'hh_inequality_state',
    'hh_poverty', 'hh_poverty_state', 'hies_state', 'cpi_annual',
    'cpi_headline', 'cpi_headline_inflation', 'cpi_state', 'cpi_lowincome'
]

print()
print('=' * 55)
loaded  = 0
missing = []
for name, df in zip(all_names, all_vars):
    if df is not None:
        yr_info = ''
        if 'year' in df.columns:
            yr_info = f'  {int(df["year"].min())}–{int(df["year"].max())}'
        print(f'  ✅  {name:<35} {df.shape[0]:>7,} rows{yr_info}')
        loaded += 1
    else:
        print(f'  ❌  {name:<35} MISSING')
        missing.append(name)

print()
print('=' * 55)
print(f'  {loaded}/21 datasets loaded')
if missing:
    print(f'  ⚠️  Missing ({len(missing)}): {missing}')
    print('      Fix: Run 00_downloadDataFinal.ipynb → 01_EDA___Data_Cleaning.ipynb')
else:
    print('  🎉  All 21 datasets loaded successfully!')
print(f'  Dashboard folder: {os.path.abspath(DASH_DIR)}')
print()
print('Run Cell 2 to export dashboard-ready files.')

Loading 21 clean datasets...

  📦  TEMA 1 — Demografi

  📦  TEMA 2 — Kos Sara Hidup

  ✅  population_malaysia                  92,000 rows  1970–1981
  ✅  population_state                    1,807,000 rows  2020–2020
  ✅  population_district                 10,567,000 rows  2020–2021
  ✅  fertility                               528 rows  1958–2023
  ✅  fertility_state                     2,280,000 rows  2001–2023
  ✅  births_annual                            24 rows  2000–2023
  ✅  deaths                                   24 rows  2000–2023
  ✅  marriages                                12 rows  2017–2022
  ✅  hh_profile                               19 rows  1970–2024
  ✅  hh_income                                21 rows  1970–2022
  ✅  hh_income_state                         303 rows  1970–2022
  ✅  hh_inequality                            20 rows  1970–2022
  ✅  hh_inequality_state                     273 rows  1974–2022
  ✅  hh_poverty                               20 rows  1970–202

In [7]:
# ============================================================
# CELL 2 — EXPORT 12 DASHBOARD-READY CSV FILES
# Depends on Cell 1 variables. Run Cell 1 first.
# ============================================================

print('Exporting dashboard-ready files...')
print('=' * 60)
export_log = []

# ── FILE 1: Population growth line ──────────────────────────
# pop_malaysia cols: year, age, sex, ethnicity, population
if pop_malaysia is not None:
    pop_total = pop_malaysia[
        (pop_malaysia['age']       == 'overall') &
        (pop_malaysia['sex']       == 'both')    &
        (pop_malaysia['ethnicity'] == 'overall')
    ][['year', 'population']].sort_values('year').copy()
    pop_total.columns = ['Year', 'Population_Thousands']
    pop_total.to_csv(f'{DASH_DIR}/dash_population_total.csv', index=False)
    print(f'  ✅  dash_population_total.csv    {len(pop_total)} rows | {int(pop_total["Year"].min())}–{int(pop_total["Year"].max())}')
    export_log.append(('dash_population_total.csv', 'Population Growth', 'Tema 1'))

# ── FILE 2: Population pyramid (all years for animated slider) ──
# Excludes 'overall' and legacy '70+' age label; keeps 17 standard bins
if pop_malaysia is not None:
    age_order = [
        '0-4','5-9','10-14','15-19','20-24','25-29','30-34',
        '35-39','40-44','45-49','50-54','55-59','60-64',
        '65-69','70-74','75-79','80+'
    ]
    age_order = [a for a in age_order if a in pop_malaysia['age'].unique()]
    pyramid   = pop_malaysia[
        (pop_malaysia['ethnicity'] == 'overall') &
        (pop_malaysia['sex'].isin(['male', 'female'])) &
        (pop_malaysia['age'].isin(age_order))
    ][['year', 'age', 'sex', 'population']].copy()
    pyramid.columns = ['Year', 'Age_Group', 'Sex', 'Population_Thousands']
    # Males go left (negative) on the pyramid chart
    pyramid['Population_Display'] = pyramid.apply(
        lambda r: -r['Population_Thousands'] if r['Sex'] == 'male' else r['Population_Thousands'],
        axis=1
    )
    pyramid.to_csv(f'{DASH_DIR}/dash_pyramid.csv', index=False)
    print(f'  ✅  dash_pyramid.csv             {len(pyramid)} rows | All years, {len(age_order)} age bands')
    export_log.append(('dash_pyramid.csv', 'Population Pyramid', 'Tema 1'))

# ── FILE 3a: National TFR trend ─────────────────────────────
# fertility cols: year, age_group, fertility_rate  — filter age_group == 'tfr'
if fertility is not None:
    tfr_nat = fertility[fertility['age_group'] == 'tfr'][['year', 'fertility_rate']].copy()
    tfr_nat.columns = ['Year', 'TFR_National']
    tfr_nat['Below_Replacement'] = tfr_nat['TFR_National'].apply(
        lambda x: 'Below 2.1' if x < 2.1 else 'At/Above 2.1'
    )
    tfr_nat = tfr_nat.sort_values('Year')
    tfr_nat.to_csv(f'{DASH_DIR}/dash_fertility_national.csv', index=False)
    print(f'  ✅  dash_fertility_national.csv  {len(tfr_nat)} rows | {int(tfr_nat["Year"].min())}–{int(tfr_nat["Year"].max())}')
    export_log.append(('dash_fertility_national.csv', 'TFR Trend', 'Tema 1'))

# ── FILE 3b: TFR by state (latest year) ─────────────────────
# fertility_state cols: year, state, age_group, fertility_rate
# Filter age_group == 'tfr'; if sex/ethnicity cols exist keep 'both'/'overall'
if fertility_state is not None:
    tfr_st = fertility_state[fertility_state['age_group'] == 'tfr'].copy()
    for col in ['sex', 'ethnicity']:
        if col in tfr_st.columns:
            vals = tfr_st[col].unique()
            if 'both' in vals:
                tfr_st = tfr_st[tfr_st[col] == 'both']
            elif 'overall' in vals:
                tfr_st = tfr_st[tfr_st[col] == 'overall']
    latest_yr  = int(tfr_st['year'].max())
    tfr_latest = (
        tfr_st[tfr_st['year'] == latest_yr]
        .groupby('state', as_index=False)['fertility_rate']
        .mean()
        .sort_values('fertility_rate', ascending=False)
    )
    tfr_latest.columns = ['State', 'TFR']
    tfr_latest['Year']   = latest_yr
    tfr_latest['Status'] = tfr_latest['TFR'].apply(
        lambda x: 'Below 2.1' if x < 2.1 else 'At/Above 2.1'
    )
    tfr_latest.to_csv(f'{DASH_DIR}/dash_fertility_state.csv', index=False)
    print(f'  ✅  dash_fertility_state.csv     {len(tfr_latest)} states | Year: {latest_yr}')
    export_log.append(('dash_fertility_state.csv', 'TFR by State', 'Tema 1'))

# ── FILE 4: Births vs Deaths ─────────────────────────────────
# births cols: year, abs (+ sex, but we use sex-total rows)
# deaths cols: year, abs
# Both datasets already contain annual totals in 'abs' column
if births is not None and deaths is not None:
    b = births[['year', 'abs']].copy()
    b.columns  = ['Year', 'Count']
    b['Category'] = 'Live Births'
    d = deaths[['year', 'abs']].copy()
    d.columns  = ['Year', 'Count']
    d['Category'] = 'Deaths'
    bd = pd.concat([b, d], ignore_index=True).sort_values(['Year', 'Category'])
    bd.to_csv(f'{DASH_DIR}/dash_births_deaths.csv', index=False)
    print(f'  ✅  dash_births_deaths.csv       {len(bd)} rows | Births & Deaths combined')
    export_log.append(('dash_births_deaths.csv', 'Births vs Deaths', 'Tema 1'))

# ── FILE 5: Ageing Index (derived metric) ───────────────────
# Ageing Index = (pop 60+) / (pop 0–14) × 100
# ≥ 100 means more elderly than children — aged nation
if pop_malaysia is not None:
    pop_mx = pop_malaysia[
        (pop_malaysia['sex']       == 'both') &
        (pop_malaysia['ethnicity'] == 'overall') &
        (pop_malaysia['age']       != 'overall')
    ].copy()
    young_ages   = ['0-4', '5-9', '10-14']
    elderly_ages = ['60-64', '65-69', '70-74', '75-79', '80+']
    young   = pop_mx[pop_mx['age'].isin(young_ages)].groupby('year')['population'].sum().reset_index()
    elderly = pop_mx[pop_mx['age'].isin(elderly_ages)].groupby('year')['population'].sum().reset_index()
    ageing  = pd.merge(young, elderly, on='year', suffixes=('_young', '_elderly'))
    ageing['Year']         = ageing['year']
    ageing['Pop_Young']    = ageing['population_young']
    ageing['Pop_Elderly']  = ageing['population_elderly']
    ageing['Ageing_Index'] = (ageing['Pop_Elderly'] / ageing['Pop_Young']) * 100
    ageing['Status']       = ageing['Ageing_Index'].apply(
        lambda x: 'Ageing Nation (≥100)' if x >= 100 else 'Pre-Ageing (<100)'
    )
    ageing[['Year', 'Pop_Young', 'Pop_Elderly', 'Ageing_Index', 'Status']].sort_values('Year').to_csv(
        f'{DASH_DIR}/dash_ageing_index.csv', index=False
    )
    print(f'  ✅  dash_ageing_index.csv        {len(ageing)} rows | Derived metric')
    export_log.append(('dash_ageing_index.csv', 'Ageing Index', 'Tema 1'))

# ── FILE 6: HERO — Median Income vs CPI Inflation ───────────
# hh_income cols: year, income_median, income_mean  (HIES survey cycles)
# cpi_inflation cols: date, division, inflation_yoy  + derived 'year'
# Merge on year; pct_change() on HIES survey gaps ≠ annual, so label clearly
if hh_income is not None and cpi_inflation is not None:
    inc = hh_income[['year', 'income_median', 'income_mean']].copy()
    inc.columns = ['Year', 'Income_Median_RM', 'Income_Mean_RM']
    inc = inc.sort_values('Year')

    cpi_ov = cpi_inflation[
        (cpi_inflation['division'] == 'overall') &
        (cpi_inflation['inflation_yoy'].notna())
    ].groupby('year')['inflation_yoy'].mean().reset_index()
    cpi_ov.columns = ['Year', 'CPI_Inflation_Pct']

    merged = pd.merge(inc, cpi_ov, on='Year', how='outer').sort_values('Year')
    # pct_change across HIES survey gaps (every 2–3 yrs) — label as inter-survey growth
    merged['Income_Growth_Pct'] = merged['Income_Median_RM'].pct_change() * 100
    merged['Real_Gain_Pct']     = merged['Income_Growth_Pct'] - merged['CPI_Inflation_Pct']
    merged['Real_Gain_Label']   = merged['Real_Gain_Pct'].apply(
        lambda x: 'Income beat inflation' if pd.notna(x) and x > 0
                  else ('Inflation won' if pd.notna(x) else 'No data')
    )
    merged.to_csv(f'{DASH_DIR}/dash_income_cpi.csv', index=False)
    print(f'  ✅  dash_income_cpi.csv          {len(merged)} rows | HERO chart data')
    export_log.append(('dash_income_cpi.csv', 'Income vs CPI HERO', 'Tema 2'))

# ── FILE 7: Income by State (all survey years) ───────────────
# hh_income_state cols: year, state, income_median, income_mean, (others)
if hh_income_state is not None:
    inc_st = hh_income_state.copy()
    # Title-case column names for Power BI / Tableau friendliness
    inc_st.columns = [
        c.replace('_', ' ').title().replace(' ', '_')
        for c in inc_st.columns
    ]
    inc_st.to_csv(f'{DASH_DIR}/dash_income_state.csv', index=False)
    print(f'  ✅  dash_income_state.csv        {len(inc_st)} rows | Income by state, all years')
    export_log.append(('dash_income_state.csv', 'Income by State', 'Tema 2'))

# ── FILE 8a & 8b: Gini national + by state ───────────────────
# hh_inequality cols: year, gini
# hh_inequality_state cols: year, state, gini
if hh_inequality is not None:
    gini_nat = hh_inequality[['year', 'gini']].copy()
    gini_nat.columns = ['Year', 'Gini_National']
    gini_nat = gini_nat.sort_values('Year')
    gini_nat.to_csv(f'{DASH_DIR}/dash_gini_national.csv', index=False)
    print(f'  ✅  dash_gini_national.csv       {len(gini_nat)} rows | National Gini')
    export_log.append(('dash_gini_national.csv', 'Gini National', 'Tema 2'))

if hh_inequality_state is not None:
    # Columns in raw data: year, state, gini (confirmed in 01_EDA Cell 6D)
    gini_st = hh_inequality_state[['year', 'state', 'gini']].copy()
    gini_st.columns = ['Year', 'State', 'Gini']
    gini_st = gini_st.sort_values(['Year', 'State'])
    gini_st.to_csv(f'{DASH_DIR}/dash_gini_state.csv', index=False)
    print(f'  ✅  dash_gini_state.csv          {len(gini_st)} rows | Gini by state (heatmap-ready)')
    export_log.append(('dash_gini_state.csv', 'Gini Heatmap', 'Tema 2'))

# ── FILE 9: Poverty trend ────────────────────────────────────
# hh_poverty cols: year, poverty_absolute, poverty_hardcore
# NOTE: poverty_hardcore has NaN for early years — kept intentionally
if hh_poverty is not None:
    pov = hh_poverty[['year', 'poverty_absolute', 'poverty_hardcore']].copy()
    pov.columns = ['Year', 'Poverty_Absolute_Pct', 'Poverty_Hardcore_Pct']
    pov = pov.sort_values('Year')
    pov.to_csv(f'{DASH_DIR}/dash_poverty.csv', index=False)
    print(f'  ✅  dash_poverty.csv             {len(pov)} rows | Absolute & hardcore poverty')
    export_log.append(('dash_poverty.csv', 'Poverty Trend', 'Tema 2'))

# ── FILE 10: CPI by category (annual average) ───────────────
# cpi_inflation cols: date, year (derived), division, inflation_yoy
# division codes: 'overall','01','02',...,'13','14' (confirmed in Chart 11)
if cpi_inflation is not None:
    div_map = {
        'overall' : 'Overall CPI',
        '01'      : 'Food & Non-Alcoholic Beverages',
        '02'      : 'Alcoholic Beverages & Tobacco',
        '03'      : 'Clothing & Footwear',
        '04'      : 'Housing, Water & Energy',
        '05'      : 'Furnishings & Household Equipment',
        '06'      : 'Health',
        '07'      : 'Transport',
        '08'      : 'Communication',
        '09'      : 'Recreation & Culture',
        '10'      : 'Education',
        '11'      : 'Food Service & Accommodation',
        '12'      : 'Financial Services',
        '13'      : 'Restaurants & Hotels',
        '14'      : 'Miscellaneous',
    }
    avail = cpi_inflation['division'].unique().tolist()
    cpi_c = cpi_inflation[cpi_inflation['division'].isin(div_map.keys())].copy()
    cpi_c['Category'] = cpi_c['division'].map(div_map)
    cpi_ann = (
        cpi_c
        .dropna(subset=['inflation_yoy'])
        .groupby(['year', 'Category'])['inflation_yoy']
        .mean()
        .reset_index()
    )
    cpi_ann.columns = ['Year', 'Category', 'Inflation_Pct_YoY']
    cpi_ann = cpi_ann.sort_values(['Year', 'Category'])
    cpi_ann.to_csv(f'{DASH_DIR}/dash_cpi_category.csv', index=False)
    print(f'  ✅  dash_cpi_category.csv        {len(cpi_ann)} rows | {cpi_ann["Category"].nunique()} categories, annual avg')
    export_log.append(('dash_cpi_category.csv', 'CPI by Category', 'Tema 2'))

# ── FILE 11: Bubble chart — Income vs Gini vs Population ────
# hies_state cols: year, state, income_median, gini, (expenditure_mean, etc.)
# Merges pop_state for bubble size; falls back to income_median if pop unavailable
if hies_state is not None:
    latest_hies = int(hies_state['year'].max())
    bubble      = hies_state[hies_state['year'] == latest_hies].copy()

    # Merge population (sex=both, ethnicity=overall, age=overall) — same fallback
    # chain used in Chart 12 of 02_analysisFinal
    pop_st = pop_state[
        (pop_state['sex']       == 'both') &
        (pop_state['ethnicity'] == 'overall') &
        (pop_state['age']       == 'overall')
    ].copy()
    if len(pop_st) == 0:
        pop_st = pop_state[
            (pop_state['sex'] == 'both') &
            (pop_state['age'] == 'overall')
        ].copy()
    if len(pop_st) == 0:
        pop_st = (
            pop_state[pop_state['sex'] == 'both']
            .groupby(['state', 'year'])['population']
            .sum()
            .reset_index()
        )

    if len(pop_st) > 0:
        pop_yr       = int(pop_st['year'].max())
        pop_by_state = pop_st[pop_st['year'] == pop_yr][['state', 'population']].copy()
        bubble       = pd.merge(bubble, pop_by_state, on='state', how='left')
    else:
        bubble['population'] = np.nan

    bubble = bubble.dropna(subset=['income_median', 'gini'])
    bubble['Year'] = latest_hies
    bubble.to_csv(f'{DASH_DIR}/dash_bubble.csv', index=False)
    print(f'  ✅  dash_bubble.csv              {len(bubble)} states | HIES {latest_hies} snapshot')
    export_log.append(('dash_bubble.csv', 'Bubble Chart (Income × Gini × Pop)', 'Combined'))

# ── SUMMARY ──────────────────────────────────────────────────
print()
print('=' * 60)
print(f'  {len(export_log)}/12 dashboard files exported')
print(f'  Location: {os.path.abspath(DASH_DIR)}')
print()
if len(export_log) < 12:
    missing_count = 12 - len(export_log)
    print(f'  ⚠️  {missing_count} file(s) skipped — source dataset was None.')
    print('      Run 00 → 01 to regenerate missing clean CSVs.')
    print()
print('  FILES:')
for fname, desc, tema in export_log:
    print(f'    {tema:<10}  {fname:<42}  {desc}')
print()
print('Run Cell 3 for Power BI step-by-step guide.')

Exporting dashboard-ready files...
  ✅  dash_population_total.csv    1104 rows | 1970–1981
  ✅  dash_pyramid.csv             31648 rows | All years, 17 age bands
  ✅  dash_fertility_national.csv  66 rows | 1958–2023
  ✅  dash_fertility_state.csv     16 states | Year: 2023
  ✅  dash_births_deaths.csv       48 rows | Births & Deaths combined
  ✅  dash_ageing_index.csv        12 rows | Derived metric
  ✅  dash_income_cpi.csv          37 rows | HERO chart data
  ✅  dash_income_state.csv        303 rows | Income by state, all years
  ✅  dash_gini_national.csv       20 rows | National Gini
  ✅  dash_gini_state.csv          273 rows | Gini by state (heatmap-ready)
  ✅  dash_poverty.csv             20 rows | Absolute & hardcore poverty
  ✅  dash_cpi_category.csv        85 rows | 4 categories, annual avg
  ✅  dash_bubble.csv              16 states | HIES 2022 snapshot

  13/12 dashboard files exported
  Location: C:\Users\nizza\Documents\my-projects\malaysia-socioeconomic-portfolio\data\dashboa

In [8]:
pbi = """
POWER BI STEP-BY-STEP GUIDE
Malaysia Socioeconomic Portrait
=====================================================================

STEP A — LOAD DATA
-------------------
1. Open Power BI Desktop
2. Home > Get Data > Text/CSV
3. Import all 13 CSV files from ../data/dashboard/
4. First Row as Headers = ON | Delimiter = Comma | Encoding = UTF-8
5. In Power Query: set Year columns to Whole Number, _RM and _Pct
   columns to Decimal Number
6. Click 'Close & Apply'

STEP B — THEME COLOURS
-----------------------
View > Themes > Customize:
  Primary   : #1B3A6B  (dark navy)
  Secondary : #0E7490  (teal / growth)
  Accent 1  : #B91C1C  (red / warning)
  Accent 2  : #15803D  (green / positive)
  Accent 3  : #7C3AED  (purple / derived)
  Background: #F8FAFC
  Text      : #1F2937

====================================================================
PAGE 1 — TEMA 1: THE AGEING NATION  [Canvas 1280 x 720]
====================================================================

HEADER TEXT BOX (full width, height 50px):
  'TEMA 1: ANJAKAN DEMOGRAFI — The Ageing Nation Story'
  Font: 18pt Bold | Background: #1B3A6B | Text: White

KPI CARDS (4 cards, row under header):
  Card 1: Latest Population    Field: Max(Population_Thousands) [latest Year]
  Card 2: Latest TFR           Field: Min(TFR_National) [latest Year]
  Card 3: Latest Ageing Index  Field: Max(Ageing_Index) [latest Year]
  Card 4: Latest TFR Status    Field: First(Below_Replacement) [latest Year]

VISUAL 1 — Population Pyramid (left, 55% width)
  Type    : Clustered Bar Chart (horizontal)
  Data    : dash_pyramid.csv
  Y-axis  : Age_Group
  Sort Y  : Custom — 0-4 at BOTTOM, 80+ at TOP (right-click axis > Sort)
  X-axis  : Sum(Population_Display)   ← already negative for Male
  Legend  : Sex
  Colors  : Male=#0E7490 | Female=#BE185D
  Slicer  : Add Year slicer (Slider type) linked to this visual only
  X-axis  : Remove the negative sign from axis labels:
            Format pane > X-axis > Custom format: #,0;#,0
  Title   : 'Population Pyramid — Use Year Slider Below'
  NOTE    : Population_Display column is already sign-encoded
            (male = negative, female = positive) from Cell 2 export

VISUAL 2 — TFR Trend Line (right, 45% width)
  Type    : Line Chart
  Data    : dash_fertility_national.csv
  X-axis  : Year
  Y-axis  : TFR_National
  Color   : #B45309
  Analytics > Constant Line: Value = 2.1
    Line color = Red | Dashed | Label = 'Replacement Level 2.1'
  Title   : 'Total Fertility Rate (TFR) — 1958 to Present'

VISUAL 3 — Births vs Deaths (bottom left)
  Type    : Area Chart
  Data    : dash_births_deaths.csv
  X-axis  : Year
  Y-axis  : Count
  Legend  : Category
  Colors  : Live Births=#15803D | Deaths=#B91C1C
  Title   : 'Births vs Deaths — Natural Growth Is Slowing'

VISUAL 4 — Ageing Index (bottom right)
  Type    : Line Chart
  Data    : dash_ageing_index.csv
  X-axis  : Year
  Y-axis  : Ageing_Index
  Legend  : Status (Pre-Ageing (<100)=#7C3AED | Ageing Nation (>=100)=#B91C1C)
  Analytics > Constant Line: Value = 100
    Line color = Red | Dashed | Label = 'Aged Nation Threshold'
  Title   : 'Ageing Index = (Pop 60+) / (Pop 0–14) × 100'

DAX MEASURES — PAGE 1:
  LatestPopYear =
    CALCULATE(MAX(dash_population_total[Year]))

  LatestPopulation =
    CALCULATE(
      MAX(dash_population_total[Population_Thousands]),
      dash_population_total[Year] = [LatestPopYear]
    )

  LatestTFRYear =
    CALCULATE(MAX(dash_fertility_national[Year]))

  LatestTFR =
    CALCULATE(
      MIN(dash_fertility_national[TFR_National]),
      dash_fertility_national[Year] = [LatestTFRYear]
    )

  TFR_Warning =
    IF([LatestTFR] < 2.1, "⚠ BELOW 2.1 — Ageing Signal", "✓ At/Above Replacement")

  LatestAgeingIndex =
    CALCULATE(
      MAX(dash_ageing_index[Ageing_Index]),
      dash_ageing_index[Year] = MAX(dash_ageing_index[Year])
    )

====================================================================
PAGE 2 — TEMA 2: INCOME vs INFLATION  [Canvas 1280 x 720]
====================================================================

HEADER TEXT BOX:
  'TEMA 2: KRISIS KOS SARA HIDUP — Can We Afford to Live Here?'
  Font: 18pt Bold | Background: #0E7490 | Text: White

KPI CARDS (4 cards):
  Card 1: Latest Median Income   Field: Max(Income_Median_RM) [latest Year]
  Card 2: Latest CPI Inflation   Field: Average(CPI_Inflation_Pct) [latest Year]
  Card 3: National Gini          Field: Max(Gini_National) [latest Year]
  Card 4: Poverty Rate           Field: Max(Poverty_Absolute_Pct) [latest Year]

VISUAL 5 — HERO: Income vs CPI (left, 65% width)
  Type    : Line and Clustered Column Chart
  Data    : dash_income_cpi.csv
  X-axis  : Year
  Column Y: Income_Median_RM (primary Y-axis — bars)
  Line Y  : CPI_Inflation_Pct (secondary Y-axis — line)
  To add secondary axis: drag CPI_Inflation_Pct to 'Line Y-axis' well
  Colors  : Income bars=#15803D | CPI line=#B91C1C
  NOTE    : HIES survey runs every 2–3 years so income bars will have
            gaps — this is correct, not a data error
  Title   : 'HERO: Median Income vs CPI Inflation — Does Real Income Grow?'

VISUAL 6 — Income by State (right, 35% width)
  Type    : Horizontal Bar Chart
  Data    : dash_income_state.csv
  Filter  : Year = latest available (add page-level filter)
  Y-axis  : State (sorted by Income_Median descending)
  X-axis  : Income_Median   ← column is titled 'Income_Median' after export
  Color   : Conditional formatting (Rules):
              If Income_Median >= [NationalMedian] → #15803D (Green)
              Else → #B91C1C (Red)
  Constant line at national median value (use DAX measure below)
  Title   : 'Median Household Income by State (Latest Year)'

VISUAL 7 — CPI by Category (bottom left)
  Type    : Line Chart
  Data    : dash_cpi_category.csv
  X-axis  : Year
  Y-axis  : Inflation_Pct_YoY
  Legend  : Category
  Colors  : Overall CPI=#1B3A6B | Food & Non-Alcoholic Beverages=#B91C1C
            Housing, Water & Energy=#0E7490 | Transport=#D97706 | Health=#7C3AED
  Add slicer: Category (multi-select) for user interactivity
  Analytics > Constant Line at Y=0 (grey, solid)
  Title   : 'CPI Inflation by Category (% YoY)'

VISUAL 8 — Poverty Trend (bottom right)
  Type    : Line Chart (two lines)
  Data    : dash_poverty.csv
  X-axis  : Year
  Y-axis  : Poverty_Absolute_Pct + Poverty_Hardcore_Pct
  Colors  : Absolute=#B45309 | Hardcore=#B91C1C (dashed)
  NOTE    : Poverty_Hardcore_Pct has NaN for early years — Power BI
            will render a gap in the hardcore line; this is expected
  Title   : 'Poverty Rate Reduction — 1970 to Present'

DAX MEASURES — PAGE 2:
  LatestIncomeYear =
    CALCULATE(MAX(dash_income_cpi[Year]),
              NOT(ISBLANK(dash_income_cpi[Income_Median_RM])))

  LatestMedianIncome =
    CALCULATE(
      MAX(dash_income_cpi[Income_Median_RM]),
      dash_income_cpi[Year] = [LatestIncomeYear]
    )

  LatestCPIYear =
    CALCULATE(MAX(dash_income_cpi[Year]),
              NOT(ISBLANK(dash_income_cpi[CPI_Inflation_Pct])))

  LatestCPI =
    CALCULATE(
      AVERAGE(dash_income_cpi[CPI_Inflation_Pct]),
      dash_income_cpi[Year] = [LatestCPIYear]
    )

  NationalMedian =
    CALCULATE(
      MAX(dash_income_state[Income_Median]),
      dash_income_state[Year] = MAX(dash_income_state[Year])
    )

====================================================================
PAGE 3 — COMBINED: INEQUALITY DEEP DIVE  [Canvas 1280 x 720]
====================================================================

HEADER TEXT BOX:
  'COMBINED: Inequality, Fertility & Cost of Living — The Full Story'
  Background: #7C3AED | Text: White

VISUAL 9 — Gini Heatmap FULL WIDTH (top 50% of canvas)
  Type    : Matrix Visual
  Data    : dash_gini_state.csv
  Rows    : State
  Columns : Year (set to Don't summarize — treat as discrete)
  Values  : Gini (Average)
  Sort    : State sorted by latest-year Gini descending (most unequal top)
  Conditional Formatting ON for Values cell background:
    Rules > Diverging
    Min:    0.30 → Green  #15803D
    Center: 0.40 → Yellow #FBBF24
    Max:    0.55 → Red    #B91C1C
  Show value in cell: ON (font 9pt, color #1F2937)
  Title   : 'Gini Coefficient by State and Year — Inequality Trend'

VISUAL 10 — TFR by State bar (bottom left)
  Type    : Horizontal Bar Chart
  Data    : dash_fertility_state.csv
  Y-axis  : State (sorted by TFR ascending)
  X-axis  : TFR
  Color   : Status — 'Below 2.1'=#B91C1C | 'At/Above 2.1'=#15803D
  Analytics > Constant Line: Value = 2.1, color = #D97706, dashed
  Title   : 'Fertility Rate by State (Latest Year)'

VISUAL 11 — Bubble/Scatter (bottom right)
  Type    : Scatter Chart
  Data    : dash_bubble.csv
  X-axis  : income_median   (raw column name — not title-cased in this file)
  Y-axis  : gini
  Size    : population       (first priority) OR expenditure_mean (fallback)
            NOTE: Cell 2 export merges population from pop_state;
            if population column is all-null, use expenditure_mean instead
  Legend  : state
  Title   : 'Income vs Inequality by State — Bubble Chart'

====================================================================
STEP C — EXPORT PDF
====================================================================

File > Export > Export to PDF
  Tick: All Pages (3 pages)
  Save as: Malaysia_Socioeconomic_Dashboard_PowerBI.pdf

BONUS — Publish to Power BI Service (free):
  Home > Publish > Select 'My Workspace'
  After publish: open browser > copy URL
  Paste link in README.md
"""

print(pbi)


POWER BI STEP-BY-STEP GUIDE
Malaysia Socioeconomic Portrait

STEP A — LOAD DATA
-------------------
1. Open Power BI Desktop
2. Home > Get Data > Text/CSV
3. Import all 13 CSV files from ../data/dashboard/
4. First Row as Headers = ON | Delimiter = Comma | Encoding = UTF-8
5. In Power Query: set Year columns to Whole Number, _RM and _Pct
   columns to Decimal Number
6. Click 'Close & Apply'

STEP B — THEME COLOURS
-----------------------
View > Themes > Customize:
  Primary   : #1B3A6B  (dark navy)
  Secondary : #0E7490  (teal / growth)
  Accent 1  : #B91C1C  (red / warning)
  Accent 2  : #15803D  (green / positive)
  Accent 3  : #7C3AED  (purple / derived)
  Background: #F8FAFC
  Text      : #1F2937

PAGE 1 — TEMA 1: THE AGEING NATION  [Canvas 1280 x 720]

HEADER TEXT BOX (full width, height 50px):
  'TEMA 1: ANJAKAN DEMOGRAFI — The Ageing Nation Story'
  Font: 18pt Bold | Background: #1B3A6B | Text: White

KPI CARDS (4 cards, row under header):
  Card 1: Latest Population    Field: 

In [9]:
tableau = """
TABLEAU PUBLIC STEP-BY-STEP GUIDE
Malaysia Socioeconomic Portrait
=====================================================================

STEP A — CONNECT DATA
---------------------
1. Open Tableau Public Desktop
2. Connect > Text File > select dash_income_cpi.csv (start here)
3. In the Data Source tab, click 'Add' to load remaining files
   Load all 13 CSV files from ../data/dashboard/
4. Set data types per file:
   - Year columns    → Number (Whole)
   - _RM columns     → Number (Decimal)
   - _Pct columns    → Number (Decimal)
   - State, Category → String (set geographic role: State/Province)
5. Rename connections to match CSV file names for clarity

====================================================================
CREATE 12 WORKSHEETS
====================================================================

SHEET 1 — Population Pyramid
  Data    : dash_pyramid.csv
  Rows    : Age_Group (sorted: 0-4 at BOTTOM, 80+ at TOP)
  Cols    : SUM(Population_Display)
  Color   : Sex → Male=#0E7490, Female=#BE185D
  Type    : Bar (horizontal, Marks > Bar)
  Pages   : Year (creates animation slider — drag to Pages shelf)
  Ref line: Add zero reference line on X-axis (Analytics > Reference Line)
  X labels: Format axis > Numbers > Custom: #,0;#,0 (removes minus sign)
  NOTE    : Population_Display is pre-encoded: male=negative, female=positive
  Title   : 'Population Pyramid — [<Year>]'

SHEET 2 — TFR Trend
  Data    : dash_fertility_national.csv
  Cols    : YEAR(Year) — continuous
  Rows    : AVG(TFR_National)
  Color   : Below_Replacement (Below 2.1=#B45309, At/Above 2.1=#15803D)
  Type    : Line + circle markers
  Ref line: Analytics > Reference Line > Constant > 2.1
              Style: Dashed | Color: #B91C1C | Label: 'Replacement Level 2.1'
  Title   : 'Total Fertility Rate — Malaysia (1958 to Present)'

SHEET 3 — Births vs Deaths
  Data    : dash_births_deaths.csv
  Cols    : YEAR(Year)
  Rows    : SUM(Count)
  Color   : Category (Live Births=#15803D | Deaths=#B91C1C)
  Type    : Area chart (Marks > Area)
  Title   : 'Births vs Deaths — Natural Population Growth'

SHEET 4 — Ageing Index
  Data    : dash_ageing_index.csv
  Cols    : YEAR(Year)
  Rows    : SUM(Ageing_Index)
  Color   : Status (Pre-Ageing (<100)=#7C3AED | Ageing Nation (>=100)=#B91C1C)
  Type    : Line + area fill
  Ref line: Analytics > Reference Line > Constant > 100
              Style: Dashed Red | Label: 'Aged Nation Threshold'
  Title   : 'Ageing Index = (Pop 60+) / (Pop 0–14) × 100'

SHEET 5 — TFR by State
  Data    : dash_fertility_state.csv
  Rows    : State (sorted by TFR ascending)
  Cols    : AVG(TFR)
  Color   : Status (Below 2.1=#B91C1C | At/Above 2.1=#15803D)
  Type    : Bar (horizontal)
  Ref line: Analytics > Reference Line > Constant > 2.1
              Style: Dashed | Color: #D97706 | Label: '2.1 Replacement'
  Title   : 'Fertility Rate by State (Latest Year)'

SHEET 6 — HERO: Income vs CPI (Dual Axis)
  Data    : dash_income_cpi.csv
  Cols    : YEAR(Year) — discrete (right-click > Discrete)
  Rows    : AVG(Income_Median_RM)
  Mark    : Bar (#15803D)
  Drag CPI_Inflation_Pct to right Rows axis > Dual Axis
  Right-click right axis > Synchronize Axis: OFF (scales differ)
  Right mark: Line (#B91C1C) + circle markers
  NOTE    : Income bars will have gaps (HIES survey every 2–3 yrs) — expected
  Title   : 'HERO: Median Household Income vs CPI Inflation'

SHEET 7 — Income by State
  Data    : dash_income_state.csv
  Filter  : Year = WINDOW_MAX(MAX(Year)) — latest year only
  Rows    : State (sorted by Income_Median descending)
  Cols    : AVG(Income_Median)
  Color   : Calculated field:
              Name: Income_vs_National
              Formula: IF AVG([Income_Median]) >=
                          WINDOW_AVG(AVG([Income_Median]))
                       THEN "Above National Avg"
                       ELSE "Below National Avg"
                       END
              Above=#15803D | Below=#B91C1C
  Ref line: Median of pane (Analytics > Reference Line > Pane > Median)
  Title   : 'Median Household Income by State (Latest Year)'
  NOTE    : Column in CSV is 'Income_Median' (title-cased by Cell 2 export)

SHEET 8 — Gini Heatmap
  Data    : dash_gini_state.csv
  Rows    : State (sorted by latest-year Gini, most unequal at top)
  Cols    : YEAR(Year) — discrete
  Marks   : Square | Color = AVG(Gini)
  Color   : Diverging palette (Red–Yellow–Green, reversed)
              Min: 0.30=#15803D | Center: 0.40=#FBBF24 | Max: 0.55=#B91C1C
  Labels  : Show AVG(Gini) formatted to 3 decimal places
  Sort    : State by AVG(Gini) descending for latest year
  Title   : 'Gini Coefficient Heatmap — State × Year'

SHEET 9 — Poverty Trend
  Data    : dash_poverty.csv
  Cols    : YEAR(Year)
  Rows    : Poverty_Absolute_Pct and Poverty_Hardcore_Pct (two pills)
  Type    : Absolute = Area fill (#B45309) | Hardcore = Dashed line (#B91C1C)
  NOTE    : Poverty_Hardcore_Pct has NULL for early years — Tableau will
            render a broken line; this is expected, not a data error
  Title   : 'Poverty Rate Decline — 1970 to Present'

SHEET 10 — CPI by Category
  Data    : dash_cpi_category.csv
  Cols    : YEAR(Year)
  Rows    : AVG(Inflation_Pct_YoY)
  Color   : Category (6 categories, distinct colors):
              Overall CPI=#1B3A6B | Food & Non-Alcoholic Beverages=#B91C1C
              Housing, Water & Energy=#0E7490 | Transport=#D97706
              Health=#7C3AED | Education=#065F46
  Type    : Line + markers
  Filter  : Category quick filter (multi-select, show on dashboard)
  Ref line: Analytics > Reference Line > Constant > 0 (grey solid)
  Title   : 'CPI Inflation by Category (% YoY)'

SHEET 11 — Bubble Chart
  Data    : dash_bubble.csv
  Cols    : AVG(income_median)    ← column retains original lowercase name
  Rows    : AVG(gini)
  Size    : AVG(population)       ← use if not all-null (see Cell 2 export)
              Fallback: AVG(expenditure_mean) if population is empty
  Color   : state (unique color per state — let Tableau auto-assign)
  Label   : state (display state name on each bubble)
  Marks   : Circle
  Title   : 'Income vs Inequality by State (Latest HIES Year)'

SHEET 12 — Population Growth
  Data    : dash_population_total.csv
  Cols    : YEAR(Year)
  Rows    : SUM(Population_Thousands)
  Type    : Line + area fill
  Color   : #1B3A6B (single color)
  Annotations: Annotate first and last data points (right-click point > Annotate > Mark)
  Title   : 'Malaysia Population Growth (1970–2024)'

====================================================================
ASSEMBLE 3 DASHBOARD PAGES
====================================================================

New Dashboard > Fixed Size > 1280 × 800 px

DASHBOARD 1 — 'The Ageing Nation'
  Top bar  : Text box header (navy #1B3A6B, white text)
  Left 55% : Sheet 1 — Population Pyramid
  Right 45%: Sheet 2 — TFR Trend
  Bot left : Sheet 3 — Births vs Deaths
  Bot right: Sheet 4 — Ageing Index
  Slicer   : Year filter (Pages shelf on Sheet 1 creates the slider)
  Action   : Dashboard > Actions > Highlight
              Source: Sheet 1 | Target: Sheet 2 | Field: Year

DASHBOARD 2 — 'Income vs Inflation'
  Top bar  : Text box header (teal #0E7490, white text)
  Left 65% : Sheet 6 — HERO Dual-Axis (full height)
  Right 35%: Sheet 7 — Income by State
  Bot left : Sheet 10 — CPI by Category
  Bot right: Sheet 9 — Poverty Trend
  Action   : Dashboard > Actions > Filter
              Source: Sheet 7 | Target: Sheet 10 | Field: State

DASHBOARD 3 — 'Inequality Deep Dive'
  Top bar  : Text box header (purple #7C3AED, white text)
  Top 55%  : Sheet 8 — Gini Heatmap (full width)
  Bot left : Sheet 5 — TFR by State
  Bot right: Sheet 11 — Bubble Chart
  Action   : Dashboard > Actions > Highlight
              Source: Sheet 8 | Target: Sheet 11 | Field: State

====================================================================
STEP D — PUBLISH TO TABLEAU PUBLIC
====================================================================

Server > Tableau Public > Save to Tableau Public As...
  Sign in: public.tableau.com (free account required)
  Workbook name: 'Malaysia_Socioeconomic_Portrait'
  After publish: right-click workbook in browser > Copy Link
  Paste link in README.md under 'Live Dashboard' section

SCREENSHOT FOR PDF SUBMISSION:
  In browser after publish > Download > Cropped Image (PNG)
  OR: Windows Snip & Sketch / macOS Command+Shift+4 for each dashboard
  Combine 3 screenshots into PDF using any PDF tool
  Save as: Malaysia_Socioeconomic_Dashboard_Tableau.pdf
"""

print(tableau)


TABLEAU PUBLIC STEP-BY-STEP GUIDE
Malaysia Socioeconomic Portrait

STEP A — CONNECT DATA
---------------------
1. Open Tableau Public Desktop
2. Connect > Text File > select dash_income_cpi.csv (start here)
3. In the Data Source tab, click 'Add' to load remaining files
   Load all 13 CSV files from ../data/dashboard/
4. Set data types per file:
   - Year columns    → Number (Whole)
   - _RM columns     → Number (Decimal)
   - _Pct columns    → Number (Decimal)
   - State, Category → String (set geographic role: State/Province)
5. Rename connections to match CSV file names for clarity

CREATE 12 WORKSHEETS

SHEET 1 — Population Pyramid
  Data    : dash_pyramid.csv
  Rows    : Age_Group (sorted: 0-4 at BOTTOM, 80+ at TOP)
  Cols    : SUM(Population_Display)
  Color   : Sex → Male=#0E7490, Female=#BE185D
  Type    : Bar (horizontal, Marks > Bar)
  Pages   : Year (creates animation slider — drag to Pages shelf)
  Ref line: Add zero reference line on X-axis (Analytics > Reference Line)
  X

In [10]:
checklist = """
FINAL CHECKLIST SEBELUM SUBMIT
=====================================================================

POWER BI (3 pages):
  [ ] Page 1 (Tema 1) : Pyramid, TFR trend, Births/Deaths, Ageing Index
  [ ] Page 2 (Tema 2) : HERO Income/CPI, State Income, CPI categories, Poverty
  [ ] Page 3 (Combined): Gini Heatmap, TFR by State, Bubble Chart
  [ ] KPI Cards on every page (min 4 per page)
  [ ] Consistent colour scheme applied (#1B3A6B / #0E7490 / #B91C1C / #15803D)
  [ ] Data source note on each page footer
  [ ] All DAX measures verified (no blank card values)
  [ ] Exported as PDF — all 3 pages

TABLEAU (3 dashboards):
  [ ] All 12 worksheets created and titled
  [ ] Dashboard 1 — The Ageing Nation complete
  [ ] Dashboard 2 — Income vs Inflation complete
  [ ] Dashboard 3 — Inequality Deep Dive complete
  [ ] Year slider / Pages filter works on Dashboard 1 Pyramid
  [ ] Click-to-filter action works on Dashboard 2 (State -> CPI Category)
  [ ] Highlight action works on Dashboard 3 (Heatmap -> Bubble)
  [ ] Published to Tableau Public — live URL obtained
  [ ] URL pasted into README.md

CSV DASHBOARD FILES (../data/dashboard/):
  [ ] dash_population_total.csv      (File 1)
  [ ] dash_pyramid.csv               (File 2)
  [ ] dash_fertility_national.csv    (File 3a)
  [ ] dash_fertility_state.csv       (File 3b)
  [ ] dash_births_deaths.csv         (File 4)
  [ ] dash_ageing_index.csv          (File 5)
  [ ] dash_income_cpi.csv            (File 6 — HERO)
  [ ] dash_income_state.csv          (File 7)
  [ ] dash_gini_national.csv         (File 8a)
  [ ] dash_gini_state.csv            (File 8b)
  [ ] dash_poverty.csv               (File 9)
  [ ] dash_cpi_category.csv          (File 10)
  [ ] dash_bubble.csv                (File 11)
  Total: 13 files

OUTPUTS (../outputs/):
  [ ] chart01_population_pyramid.html
  [ ] chart02_fertility_rate.html
  [ ] chart03_births_deaths_marriages.html
  [ ] chart04_fertility_by_state.html
  [ ] chart05_ageing_index.html
  [ ] chart06_income_vs_cpi_hero.html
  [ ] chart07_real_purchasing_power.html
  [ ] chart08_income_by_state.html
  [ ] chart09_gini_heatmap.html
  [ ] chart10_poverty_trend.html
  [ ] chart11_cpi_by_category.html
  [ ] chart12_bubble_gabungan.html
  Total: 12 HTML charts

SUBMISSION PACKAGE:
  [ ] GitHub repo with README (Tableau live URL + Power BI publish URL)
  [ ] Malaysia_Socioeconomic_Dashboard_PowerBI.pdf
  [ ] Malaysia_Socioeconomic_Dashboard_Tableau.pdf  (or live Tableau URL)
  [ ] All 4 notebooks submitted: 00, 01, 02, 03
  [ ] Nama dan No. Kad Pengenalan on PDF cover page

FOLDER STRUCTURE:
  malaysia-socioeconomic-portfolio/
  +-- notebooks/
  |   +-- 00_downloadDataFinal.ipynb
  |   +-- 01_EDA___Data_Cleaning.ipynb
  |   +-- 02_analysisFinal.ipynb
  |   +-- 03_dashboard.ipynb           (this notebook)
  +-- data/
  |   +-- raw/          (21 CSVs from OpenDOSM API)
  |   +-- clean/        (21 cleaned CSVs — output of notebook 01)
  |   +-- dashboard/    (13 dashboard-ready CSVs — output of notebook 03)
  +-- outputs/          (12 interactive HTML charts — output of notebook 02)
  +-- dashboards/
  |   +-- Malaysia_Dashboard.pbix
  |   +-- Malaysia_Dashboard_PowerBI.pdf
  |   +-- Malaysia_Dashboard.twbx
  |   +-- Malaysia_Dashboard_Tableau.pdf
  +-- README.md
"""

print(checklist)

# ── Visual mapping quick-reference table ─────────────────────
print('VISUAL MAPPING QUICK REFERENCE')
print('=' * 80)
print(f'  {"CHART":<40} {"CSV FILE":<35} {"TOOL"}')
print('  ' + '-' * 78)

mapping = [
    # (chart description,                     csv file,                       tool)
    ('Population Pyramid (animated slider)',   'dash_pyramid.csv',             'PBI + Tableau'),
    ('TFR National Trend Line',               'dash_fertility_national.csv',  'PBI + Tableau'),
    ('TFR by State Horizontal Bar',           'dash_fertility_state.csv',     'PBI + Tableau'),
    ('Births vs Deaths Area Chart',           'dash_births_deaths.csv',       'PBI + Tableau'),
    ('Ageing Index Trend Line',               'dash_ageing_index.csv',        'PBI + Tableau'),
    ('HERO: Income vs CPI Dual-Axis',         'dash_income_cpi.csv',          'PBI + Tableau'),
    ('Income by State Ranked Bar',            'dash_income_state.csv',        'PBI + Tableau'),
    ('Gini National Trend (KPI / line)',      'dash_gini_national.csv',       'PBI'),
    ('Gini Heatmap (State × Year)',           'dash_gini_state.csv',          'PBI + Tableau'),
    ('Poverty Rate Trend (2 lines)',          'dash_poverty.csv',             'PBI + Tableau'),
    ('CPI by Category Multi-Line',            'dash_cpi_category.csv',        'PBI + Tableau'),
    ('Bubble: Income × Gini × Population',   'dash_bubble.csv',              'PBI + Tableau'),
    ('Population Growth Line',               'dash_population_total.csv',    'PBI + Tableau'),
]

for chart, csv_f, tool in mapping:
    print(f'  {chart:<40} {csv_f:<35} {tool}')

print()
print(f'  Total: {len(mapping)} visuals across 13 CSV files')
print()
print('=' * 80)
print('Notebook 03 COMPLETE!')
print()
print('NEXT STEPS:')
print('  1. Open Power BI Desktop  -> follow CELL 3 guide')
print('  2. Open Tableau Public    -> follow CELL 4 guide')
print('  3. Export both to PDF')
print('  4. Push all notebooks + PDFs to GitHub')
print('  5. Paste live Tableau URL into README.md')


FINAL CHECKLIST SEBELUM SUBMIT

POWER BI (3 pages):
  [ ] Page 1 (Tema 1) : Pyramid, TFR trend, Births/Deaths, Ageing Index
  [ ] Page 2 (Tema 2) : HERO Income/CPI, State Income, CPI categories, Poverty
  [ ] Page 3 (Combined): Gini Heatmap, TFR by State, Bubble Chart
  [ ] KPI Cards on every page (min 4 per page)
  [ ] Consistent colour scheme applied (#1B3A6B / #0E7490 / #B91C1C / #15803D)
  [ ] Data source note on each page footer
  [ ] All DAX measures verified (no blank card values)
  [ ] Exported as PDF — all 3 pages

TABLEAU (3 dashboards):
  [ ] All 12 worksheets created and titled
  [ ] Dashboard 1 — The Ageing Nation complete
  [ ] Dashboard 2 — Income vs Inflation complete
  [ ] Dashboard 3 — Inequality Deep Dive complete
  [ ] Year slider / Pages filter works on Dashboard 1 Pyramid
  [ ] Click-to-filter action works on Dashboard 2 (State -> CPI Category)
  [ ] Highlight action works on Dashboard 3 (Heatmap -> Bubble)
  [ ] Published to Tableau Public — live URL obtained
  